# Week 5: Structured data — explore, query, and join

**Monday, October 5 · COMPSS 211A**

Last week you worked with a report on **16,876 San Francisco 311 requests**. This week the same team comes back with three questions:

1. **"Can we trust this file?"** Look through the records and decide which ones to keep.
2. **"Our data team uses SQL."** Answer a question you already know in SQL, and check the answer against pandas.
3. **"Can you add the supervisors' names?"** Join a second table that has the names, check the join, and check whether the names are still right.

By the end of today you should be able to:

- say what one row of a table represents;
- get the same answer from pandas and from SQL, including when values are missing;
- join two tables and show that the join worked.

We'll build the pandas join together. The SQL join is there for you to read and run.

## Files, tables, and databases

A few words we'll use today:

- A **CSV** is a file. A **DataFrame** is the table you work with in Python. A **database** stores tables so you can ask them questions with **SQL**. Today we use **SQLite**, a small database that comes with Python.
- A **schema** describes a table: what each column means, what type it is, and when it can be empty.
- A **key** is a column that identifies each row. Every request has its own `request_id`. The `district` column is different: many requests share a district, and that's expected.
- One row is **one request**. That isn't the same as one problem or one person: three neighbors can report the same pothole. After you group the data, one row stands for a whole group.

**Predict:** a district has ten requests. How many rows does it have in the original table? How many in a table with one row per district?

**Setup:** run the next cell once. It loads the same 311 data as Week 4, plus a small table of San Francisco's supervisor districts. You fill in the rest of the notebook as we go, so it won't run from top to bottom until you've done the exercises.

<details>
<summary>Where the data comes from</summary>

- The 311 requests were opened August 3–9, 2026, and downloaded September 22, 2026. Each request's status and closing time are as they were on the download day.
- `hours_to_close` is empty when a request wasn't closed yet, or when its closing time was earlier than its opening time. The `duration_note` column tells you which. A value of 0 hours is a real measurement, not a missing one.
- The district table comes from the city's [Supervisor Districts (2022) dataset](https://data.sfgov.org/d/f2zs-jevy), downloaded September 23, 2026. We kept the supervisor names exactly as the city publishes them. One of them is out of date; you'll find it in Section 4.
- More background: [Week 4's source notes](../week04_reproducible-analyses/week04_scripts.ipynb) and [the city's guide to the 311 data](https://sfdigitalservices.gitbook.io/dataset-explainers/311-cases). Keep in mind that these are requests people made. They don't count every problem in the city, and they don't tell you what all residents need.
</details>

In [ ]:
from pathlib import Path
import sqlite3

import pandas as pd
from IPython.display import display


def find_data(filename):
    """Find a data file in the course repository's data folder."""
    for root in [Path.cwd(), *Path.cwd().parents]:
        candidate = root / "data" / filename
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(f"Could not find {filename}. Open this notebook from the complete course repository.")


requests = pd.read_csv(find_data("sf311_requests.csv"), dtype={"request_id": "string", "district": "Int64"})
districts = pd.read_csv(find_data("sf_supervisor_districts.csv"), dtype=str)
print(len(requests), "requests;", len(districts), "districts")

## 1. Take a first look · 15 minutes

You did most of these checks in Week 4. Today we turn them into a routine you can run on any new table.

| Check | Question | pandas |
| --- | --- | --- |
| Shape and types | Are IDs stored as text and measurements as numbers? | `.shape`, `.dtypes` |
| Missing values | Which columns have gaps? | `.isna().sum()` |
| Keys | Does every request have an ID, and is each ID unique? | `.isna().sum()`, `.duplicated().sum()` |
| Categories | Are there labels you don't understand? | `.value_counts(dropna=False)` |
| Ranges and dates | Do the values make sense? Do the dates cover the right week? | `.describe()`, `pd.to_datetime(...)` |

The next cell shows the shape, the types, and the missing values for you.

In [ ]:
# Supplied: shape, types, and missing values.
print("Shape:", requests.shape)
display(pd.DataFrame({"dtype": requests.dtypes, "missing": requests.isna().sum()}))

**Your turn:** check that every `request_id` is there and that none repeats. Then look at every value of `channel`, including missing ones. Which labels would you ask the data owner about?

A unique ID means each *record* is unique. Two records can still be about the same pothole.

In [ ]:
# Your turn

**Together:** how many different days do you expect in `opened_at`? Predict, then run the cell. It also summarizes `hours_to_close` and counts how many requests have no usable duration, and why.

We cleaned `hours_to_close` in Week 4, so it has no negative values. That doesn't mean every original timestamp was right: look at `duration_note` to see what had to be left out.

In [ ]:
opened_dates = pd.to_datetime(requests["opened_at"])
print("Period:", opened_dates.min(), "to", opened_dates.max())
print("Calendar days:", opened_dates.dt.normalize().nunique())
display(requests["hours_to_close"].describe())
display(requests["duration_note"].value_counts(dropna=False))

**Decide and write it down.** 11 requests have the channel `Test`. Are they real requests? The label alone can't tell us. For today we'll use this rule, and write it down so others can check it:

- leave out the 11 `Test` requests;
- keep the 72 requests with no channel;
- keep the original `requests` table unchanged.

This is a working assumption until someone can explain the `Test` records. It's not a general rule to delete anything that looks odd.

**Your turn:** make `analysis`, a copy of the rows you keep. Write the rule out in full, so anyone reading your code can see you meant to keep the missing channels:

```python
(requests["channel"] != "Test") | requests["channel"].isna()
```

Then check that exactly 11 rows were left out, that all 72 rows with no channel are still there, and that `analysis` has **16,865 rows**.

In [ ]:
# Your turn
analysis = ...

**Write two short notes:** what did your first look find? Why is the `Test` rule only a working assumption, and what would make you change it?

_Your notes here._

## 2. The same questions in SQL · 30 minutes

In SQL you describe the table you want back, and the database builds it for you. Here's an example:

```sql
SELECT category, COUNT(*) AS requests
FROM requests
WHERE status = 'Open'
GROUP BY category
ORDER BY requests DESC, category ASC
LIMIT 5
```

Read it in this order: start with the `requests` table, keep the open requests, make one row per category, count the requests in each, sort, and keep the top five. `SELECT` comes first in the query, but it describes the finished result. The second sort key, `category ASC`, decides the order when two categories have the same count.

Each part matches something you already do in pandas:

| SQL | pandas |
| --- | --- |
| `SELECT a, b` | `df[["a", "b"]]` |
| `WHERE status = 'Open'` | `df.loc[df["status"] == "Open"]` |
| `AND`, `OR`, `NOT` | `&`, `\|`, `~`, with each condition in parentheses |
| `GROUP BY x` with `COUNT(*)` | `df.groupby("x", dropna=False).size()` |
| `COUNT(hours_to_close)` | `["hours_to_close"].count()`: only the rows that have a value (0 counts) |
| `ORDER BY n DESC` | `.sort_values("n", ascending=False)` |
| `LIMIT 5`, after sorting | `.head(5)`, after sorting |

**Watch out for missing values.** SQL puts all rows with a missing group label into one group. pandas `groupby` drops those rows unless you add `dropna=False`.

The next cell copies three tables into a small database that lives in memory:

- `raw_requests`: every record, including `Test`;
- `requests`: your `analysis` table;
- `districts`: the district table.

If you change `analysis` later, run this cell again. The database keeps its own copy and doesn't see changes to your DataFrame.

In [ ]:
# Supplied: run this again if you change `analysis`.
database = sqlite3.connect(":memory:")
requests.to_sql("raw_requests", database, index=False)
analysis.to_sql("requests", database, index=False)
districts.to_sql("districts", database, index=False)


def sql(query):
    """Run a SQL query on our in-memory database and return a DataFrame."""
    return pd.read_sql_query(query, database)


sql("SELECT request_id, category, channel, district FROM requests ORDER BY request_id LIMIT 5")

**Together:** change the example so it shows **all** categories of open requests, not just five. Do it in SQL and in pandas, and check that every category and every count match, including the total. Then show only the top five. Why check the full table before you cut it down?

In [ ]:
# Together: SQL, then pandas.

**Your turn:** in SQL, count the requests that came in by **Phone** in each district, biggest first. If two districts tie, sort them by district number. Save the result as `phone_by_district`.

Then do the same in pandas, twice: once with `groupby("district")` and once with `groupby("district", dropna=False)`. Compare every district, not just one, and check whether each total adds up to the number of Phone requests in `analysis`.

**Checkpoint:** SQL counts **81 Phone requests with no district**. Which pandas version keeps them? Comparing a single district's count would never have shown you this.

In [ ]:
# Your turn
phone_by_district = sql("""

""")
phone_by_district

**Predict, run, explain: filtering when values are missing.** This time use `raw_requests`, which still has the `Test` rows.

The first query below looks like our pandas rule. Does it keep the same rows? Predict, then run it.

After you run it: in SQL, comparing anything with a missing value (`NULL`) gives "unknown", not True. `WHERE` only keeps rows where the condition is True, so the 72 requests with no channel disappear. To keep them, you have to ask for them: add `OR channel IS NULL`.

Numbers to expect: **16,793** rows from the first query and **16,865** in our pandas table. The difference is the **72** missing channels. After your fix, check that the SQL result has exactly the same request IDs as `analysis`, not just the same number of rows.

In [ ]:
# Supplied: predict first, then run.
incomplete_selection = sql("SELECT request_id FROM raw_requests WHERE channel != 'Test'")
print("Incomplete SQL:", len(incomplete_selection), "| pandas analysis:", len(analysis))

# Together: add OR channel IS NULL, then compare the request IDs.
# repaired_selection = sql(...)
# set(repaired_selection["request_id"]) == set(analysis["request_id"])

**Two kinds of counts.** Will these two numbers be the same? Predict first.

`COUNT(*)` counts rows. `COUNT(hours_to_close)` counts only the rows that have a duration; a duration of 0 still counts. A missing duration is not a zero. This is the same difference as `size` and `count` in pandas, and a good report shows both.

In [ ]:
# Supplied: rows vs rows that have a duration.
display(sql("""
    SELECT COUNT(*) AS requests, COUNT(hours_to_close) AS with_time
    FROM requests
"""))
print("pandas:", len(analysis), "requests;", analysis["hours_to_close"].count(), "with time")

## 3. Join two tables · 25 minutes

Our requests have a district number. The supervisors' names are in a second table, `districts`. A **join** brings the two together by matching rows on a shared column, called the **key**. Here the key is the district.

Many requests belong to the same district, but each district has one row in the district table. This is a **many-to-one** join. Every request with a district should find exactly one match. Requests with no district should stay in the table, unmatched, so we can still see them.

First we check that the matching works. Then we check whether the names are right.

**Predict:** how many rows should the joined table have? Look at the type of the district column in both tables.

In [ ]:
display(districts.head(3))
print("requests['district']:", analysis["district"].dtype)
print("districts['sup_dist']:", districts["sup_dist"].dtype)

**Predict:** will the join in the next cell work? Run it and read the error.

In the requests, the district is a number (`5`). In the district table it's text (`"5"`, and `"05"` in another column). pandas won't match a number with text.

Here it's safe to turn the district text into numbers. Be careful doing that with other IDs, though: turning the ZIP code `"02139"` into a number loses the leading zero.

In [ ]:
# Supplied: we catch the error so the rest of the notebook still runs.
try:
    analysis.merge(districts, left_on="district", right_on="sup_dist")
except ValueError as error:
    print(type(error).__name__, ":", error)

**Your turn: build and check the join in pandas.**

1. Give the district table a number version of the district: `districts["district"] = pd.to_numeric(districts["sup_dist"]).astype("Int64")`. Check that no district is missing and none appears twice.
2. Join the tables and save the result as `joined`: `analysis.merge(districts, on="district", how="left", validate="many_to_one", indicator=True)`.
   - `how="left"` keeps every request, even ones that don't find a match.
   - `validate="many_to_one"` stops with an error if a district appears twice in the district table.
   - `indicator=True` adds a column, `_merge`, that says whether each request found a match.
3. Check the result: compare the number of rows and the request IDs before and after the join, and check that no request ID now appears twice. Count the values of `_merge`. What district do the unmatched requests have?

**Checkpoint:** 16,865 rows, 16,865 different request IDs, and 110 unmatched requests, all with no district. An *inner* join would have quietly dropped those 110. And a perfect row count still doesn't tell you the names are right. We'll come back to that in Section 4.

In [ ]:
# Your turn

**Together: what if the district table has a duplicate?** Predict what happens if district 5 appears twice. Run the cell and compare the number of rows with the number of different request IDs.

Every district 5 request now appears twice, so any count by district would be too high, and pandas wouldn't warn you. `validate` catches the problem before you use the result.

In [ ]:
broken_districts = pd.concat([districts, districts.loc[districts["district"] == 5]])
too_many = analysis.merge(broken_districts, on="district", how="left")
print(len(analysis), "requests became", len(too_many), "rows")
print("Distinct request IDs:", too_many["request_id"].nunique())

try:
    analysis.merge(broken_districts, on="district", how="left", validate="many_to_one")
except pd.errors.MergeError as error:
    print(type(error).__name__, ":", error)

**The same join in SQL.** Read and run this one; you don't have to write it. `LEFT JOIN ... ON ...` keeps every request, like `how="left"`. `CAST` turns the text district into a number, so the two columns match. The database uses the original district table from Section 2.

SQL has no `validate` option, so a duplicate in the district table would multiply rows here too, without a warning. Check the totals yourself.

We group by the district number *and* the name, because a name isn't a reliable ID. Check that the total is still 16,865 and that the requests with no district are still there.

In [ ]:
# Supplied: read and run.
sql_district_counts = sql("""
    SELECT r.district, d.sup_name AS supervisor_label, COUNT(*) AS requests
    FROM requests AS r
    LEFT JOIN districts AS d
        ON r.district = CAST(d.sup_dist AS INTEGER)
    GROUP BY r.district, d.sup_name
    ORDER BY requests DESC, r.district ASC
""")
display(sql_district_counts)
print("Total:", sql_district_counts["requests"].sum())

## 4. Write a report you can stand behind · 15 minutes

**Your turn:** from `joined`, make `supervisor_report` with one row per district. Group by `district` and `sup_name` with `dropna=False`, and use `.agg(...)` as in Week 4 to get:

- `requests`: the number of requests (`size`);
- `with_time`: how many of them have a duration (`count`);
- `median_hours`: the median of those durations (`median`).

Sort by `requests`, largest first, and keep the group with no district.

**Check:** `requests` should add up to **16,865** and `with_time` to **15,733**. The other **1,132** requests have no usable duration: 1,113 were never closed and 19 closed before they opened.

The median only describes the requests that have a duration. It doesn't describe every request, and it isn't a score for the supervisor: the kinds of requests, how people report them, and how many are still open all differ between districts.

**The join worked, but one name is wrong.** The district table lists Joel Engardio for District 4. According to the [Board of Supervisors' page about him](https://sfbos.archive.sf.gov/supervisor-engardio-district-4), he left office on October 18, 2025, almost a year before these requests. We downloaded the table in September 2026, but the city hadn't updated that name. We keep the name as published, so anyone can see where it came from.

**Discuss:** should the report show who represented each district when the requests came in, or who represents it now? Where would you find that information, with a date? Until you've checked, call this a **district report with names from an outdated table**, not a report by supervisor. A join can match every row correctly and still attach wrong information.

In [ ]:
# Your turn
supervisor_report = ...

## 5. Exit · 5 minutes

Answer on your own:

1. Why did the first SQL filter keep fewer rows than our pandas table? How did you fix it?
2. Give one finding from `supervisor_report`. If it's a median, say how many durations it's based on and how many requests the district has.
3. Name one check that shows your join worked, and one reason the names still need checking.

_Write your answers here._

**Looking ahead to text data:** a document ID works just like a request ID. When you join extra information to documents, such as authors or topics, rows can multiply, or documents with a missing key can drop out. Next week you'll build a table of documents from an API. Keep track of the IDs, and of which rows you kept, the same way you did today.

## Optional practice

**Interactive.** [The pandas and SQL comparison](../../docs/interactives/week05-pandas-sql.html) uses six made-up requests to show how rows go missing or get repeated. Predict each result, check it, then fix the comparison. It takes about 10 minutes. Open the file in your browser from your copy of the repository; GitHub only shows its code.

### Filter rows or filter groups?

Change your Phone query so it keeps only districts with **at least 300 Phone requests**, and leave out the requests with no district. `WHERE` picks rows *before* grouping; `HAVING` picks groups *after* counting. Start from all requests, not your top-five table.

Fill in the two blanks, then check your answer in pandas: group with `dropna=False`, then filter the counts. You should get district 9 with **366** and district 3 with **329**. Why wouldn't `WHERE COUNT(*) >= 300` work?

```sql
SELECT district, COUNT(*) AS requests
FROM requests
WHERE channel = 'Phone' AND district IS NOT NULL
GROUP BY district
HAVING __________
ORDER BY __________;
```

**Interview-style practice.** [SQL interview practice](../../docs/interactives/week05-sql-practice.html) has ten questions like the ones in analyst job interviews, on a sample of this week's 311 data. You type real SQL, run it in your browser, and the page checks your answer. It needs an internet connection.

More practice: Exponent's lessons on [missing values](https://www.tryexponent.com/courses/sql-interviews/sql-null-values), [grouping](https://www.tryexponent.com/courses/sql-interviews/sql-group-by-having), and [joins](https://www.tryexponent.com/courses/sql-interviews/sql-joins-and-duplicate-control). Our exercises use the course's own data and checked results. CTEs and window functions aren't part of this week.